In [1]:
from google.cloud import bigquery
from google.oauth2 import service_account

import pandas as pd
import pandas_gbq

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 10)

In [4]:
meu_service_account_file = r"C:\Users\1590717\Downloads\meu-projeto-eng-dados-512912-53c59e4b959b.json"
meu_gcp_project_id = "meu-projeto-eng-dados-512912"

my_gcp_credentials = service_account.Credentials.from_service_account_file(meu_service_account_file)
my_gcp_client = bigquery.Client(credentials=my_gcp_credentials)

In [5]:
df_silver = my_gcp_client.query(
    f"SELECT * FROM silver.complete_dc_marvel",
).to_dataframe(create_bqstorage_client=False)

In [10]:
dim_film = (
    df_silver[["FILM", "GENRE", "RATING_MPA", "LANGUAGE", "BREAK_EVEN", "MCU", "MALE_FEMALE_LED", "PHASE"]]
    .drop_duplicates()
    .sort_values(by="FILM")
    .reset_index(drop=True)
)

dim_film["ID_FILM"] = dim_film.index + 1

dim_film.head(5)

,FILM,GENRE,RATING_MPA,LANGUAGE,BREAK_EVEN,MCU,MALE_FEMALE_LED,PHASE,ID_FILM
0,Ant-Man,Superhero,PG-13,English,Success,True,Male,2.0,1
1,Ant-Man and the Wasp,Superhero,PG-13,English,Success,True,Co-starring,3.0,2
2,Ant-Man and the Wasp: Quantumania,Superhero,PG-13,English,Flop,True,Co-starring,5.0,3
3,Aquaman,Globetrotting Adventure,PG-13,English,Success,False,Male,NaN,4
4,Aquaman and the Lost Kingdom,Action Epic,PG-13,English,Flop,False,Male,NaN,5


In [12]:
dim_staff = (
    df_silver[["DIRECTOR", "WRITER", "STAR"]]
    .drop_duplicates()
    .sort_values(by=["DIRECTOR", "WRITER"])
    .reset_index(drop=True)
)

dim_staff["ID_STAFF"] = dim_staff.index + 1

dim_staff

,DIRECTOR,WRITER,STAR,ID_STAFF
0,Alan Taylor,Christopher L Yost,"Chris Hemsworth, Natalie Portman, Tom Hiddleston",1
1,Andy Muschietti,Christina Hodson,"Ezra Miller, Michael Keaton, Sasha Calle",2
2,Andy Serkis,Kelly Marcel,"Tom Hardy, Woody Harrelson, Michelle Williams",3
3,Ang Lee,Stan Lee,"Eric Bana, Jennifer Connelly, Sam Elliott",4
4,Angel Manuel Soto,Gareth DunnetAlcocer,"Xolo Mariduea, Bruna Marquezine, Becky G",5
...,...,...,...,...
99,Zack Snyder,Bob Kane,"Ben Affleck, Henry Cavill, Amy Adams",100
100,Zack Snyder,Dave Gibbons,"Jackie Earle Haley, Patrick Wilson, Carla Gugino",101
101,Zack Snyder,David S Goyer,"Henry Cavill, Amy Adams, Michael Shannon",102
102,Zack Snyder,Jerry Siegel,"Ben Affleck, Gal Gadot, Jason Momoa",103


In [14]:
dim_franchise = (
        df_silver[["FRANCHISE", "CHARACTER_FAMILY"]]
        .drop_duplicates()
        .sort_values(by=["FRANCHISE","CHARACTER_FAMILY" ])
        .reset_index(drop=True)

)

dim_franchise["ID_FRANCHISE"]= dim_franchise.index + 1

dim_franchise

,FRANCHISE,CHARACTER_FAMILY,ID_FRANCHISE
0,DC,Aquaman,1
1,DC,Batman,2
2,DC,Birds of Prey,3
3,DC,Blue Beetle,4
4,DC,Constantine,5
...,...,...,...
33,Marvel,Shang-Chi,34
34,Marvel,Spider-Man,35
35,Marvel,Spider-Man Allies and Villains,36
36,Marvel,Thor,37


In [16]:
dim_production = (
        df_silver[["PRODUCTION_COMPANY", "DISTRIBUTOR", "COUNTRY_ORIGIN", "FILMING_LOCATION"]]
        .drop_duplicates()
        .sort_values(by=["PRODUCTION_COMPANY", "DISTRIBUTOR", "COUNTRY_ORIGIN", "FILMING_LOCATION"])
        .reset_index(drop=True)

)

dim_production["ID_PRODUCTION"]= dim_production.index + 1

dim_production

,PRODUCTION_COMPANY,DISTRIBUTOR,COUNTRY_ORIGIN,FILMING_LOCATION,ID_PRODUCTION
0,20th Century Studios,20th Century Studios,United States,"Medfield State Hospital 45 Hospital Road, Med...",1
1,Amen Ra Films,New Line Cinema,United States,"Vancouver, British Columbia, Canada",2
2,Animal Logic,Warner Bros.,United States,"Disney Studios, Moore Park, Sydney, New South ...",3
3,Artistry Limited,Tri-Star Pictures,United Kingdom,"Chicago, Illinois, USA",4
4,Atlas Entertainment,Warner Bros.,United States,"Fuerteventura, Canary Islands, Spain",5
...,...,...,...,...,...
102,Warner Bros Pictures Animation,Warner Bros.,United States,"Washington, District of Columbia, USA",103
103,NaN,Paramount Pictures,NaN,NaN,104
104,NaN,Sony Pictures Entertainment,NaN,NaN,105
105,NaN,Twentieth Century Fox,NaN,NaN,106


In [18]:
dim_date= df_silver[["US_RELEASE_DATE"]].dropna().drop_duplicates().sort_values(by="US_RELEASE_DATE").reset_index(drop=True)
   
dim_date["YEAR"]= dim_date["US_RELEASE_DATE"].dt.year
dim_date["MONTH"]= dim_date["US_RELEASE_DATE"].dt.month
dim_date["QUARTER"]= dim_date["US_RELEASE_DATE"].dt.quarter


dim_date["ID_DATE"]= dim_date["US_RELEASE_DATE"].dt.strftime("%Y%m%d").astype(int)

dim_date


,US_RELEASE_DATE,YEAR,MONTH,QUARTER,ID_DATE
0,1978-12-15,1978,12,4,19781215
1,1981-06-19,1981,6,2,19810619
2,1983-06-17,1983,6,2,19830617
3,1984-11-21,1984,11,4,19841121
4,1986-08-01,1986,8,3,19860801
...,...,...,...,...,...
108,2023-06-16,2023,6,2,20230616
109,2023-08-18,2023,8,3,20230818
110,2023-11-10,2023,11,4,20231110
111,2023-12-20,2023,12,4,20231220


In [19]:
fact_movies = df_silver.copy()


In [20]:
fact_movies = fact_movies.merge(
    dim_film,
    on=["FILM", "GENRE", "RATING_MPA", "LANGUAGE", "BREAK_EVEN", "MCU", "MALE_FEMALE_LED", "PHASE"],
    how="left",
)

In [21]:
fact_movies = fact_movies.merge(
    dim_staff,
    on=["DIRECTOR", "WRITER", "STAR"],
    how="left",
)

In [22]:
fact_movies = fact_movies.merge(
    dim_franchise,
    on=["FRANCHISE", "CHARACTER_FAMILY"],
    how="left",
)

In [23]:
fact_movies = fact_movies.merge(
    dim_production,
    on=["PRODUCTION_COMPANY", "DISTRIBUTOR", "COUNTRY_ORIGIN", "FILMING_LOCATION"],
    how="left",
)

In [24]:
fact_movies = fact_movies.merge(
    dim_date,
    on=["US_RELEASE_DATE"],
    how="left",
)

In [25]:
fact_movies = fact_movies[
    [
        "ID_FILM", "ID_STAFF", "ID_FRANCHISE", "ID_PRODUCTION",
        "ID_DATE", "BUDGET_x", "INFLATION_ADJUSTED_BUDGET",
        "GROSS_WORLD_WIDE", "INFLATION_ADJUSTED_WORLDWIDE_GROSS",
        "GROSS_US_CANADA", "DOMESTIC", "BOX_OFFICE_GROSS_OTHER_TERRITORIES",
        "GROSS_OPENING_WEEKEND", "GROSS_TO_BUDGET",
        "RATING_IMDB", "ROTTEN_TOMATOES_CRITIC_SCORE",
        "VOTE", "OSCAR", "NOMINATION", "WIN", "MINUTES"
    ]
]

In [28]:
minhas_tabelas = {
    "dim_date": dim_date,
    "dim_film": dim_film,
    "dim_franchise": dim_franchise,
    "dim_production": dim_production,
    "dim_staff": dim_staff,
    "fact_movies": fact_movies,
}

for key, value in minhas_tabelas.items():
    print(key)

    pandas_gbq.to_gbq(
        value,
        project_id=meu_gcp_project_id,
        destination_table=f"gold.{key}",
        if_exists="replace",
        credentials=my_gcp_credentials,
    )

dim_date
dim_film
dim_franchise
dim_production
dim_staff
fact_movies
